In [25]:
import kagglehub
import pandas as pd
import os

# הורדת מאגר אירועי Windows
path = kagglehub.dataset_download("mehulkatara/windows-event-log")

# טעינת קובץ הלוג
target_file = "eventlog.csv"
df = pd.read_csv(os.path.join(path, target_file))

# הצגת הנתונים הגולמיים לפני הקידוד
print("Raw Windows Logs:")
print(df.head())

# הגדרת העמודות שמרכיבות את הפעולה הייחודית
# (יש לוודא ששמות העמודות תואמים בדיוק לפלט של ההדפסה הקודמת, למשל 'Source' ו-'Event ID')
features = ['Source', 'Computer']

# הפעלת הפונקציה שלך שממירה את המחרוזות ל-IDs כרונולוגיים
event_sequence, vocab = build_event_vocabulary(df, features)
vocab_size = len(vocab)
print(f"\nVocabulary size built from raw strings: {vocab_size} unique events.")

# יצירת חלונות הזמן (BPTT Window) לאימון ה-LSTM
seq_length = 10
X_train, y_train = creating_training_sequence(event_sequence, seq_length=seq_length)

print(f"Training sequences shape ready for model: {X_train.shape}")

100%|██████████| 6.80M/6.80M [00:00<00:00, 31.2MB/s]

Extracting files...


Raw Windows Logs:
   Unnamed: 0      MachineName          Category    EntryType  \
0           1  LAPTOP-1MKMTVPM               (0)  Information   
1           2  LAPTOP-1MKMTVPM  Logging/Recovery        Error   
2           3  LAPTOP-1MKMTVPM  Logging/Recovery        Error   
3           4  LAPTOP-1MKMTVPM               (0)  Information   
4           5  LAPTOP-1MKMTVPM               (0)  Information   

                                             Message  \
0  Successfully scheduled Software Protection ser...   
1  svchost (13360,R,98) TILEREPOSITORYS-1-5-18: E...   
2  svchost (15040,R,98) TILEREPOSITORYS-1-5-18: E...   
3  Successfully scheduled Software Protection ser...   
4  Successfully created restore point (Process = ...   

                                 Source        TimeGenerated country  \
0  Software Protection Platform Service  2020-11-14 08:41:59   India   
1                                 ESENT  2020-11-14 08:25:14   India   
2                                 ESEN

KeyError: "['Computer'] not in index"

In [18]:
#library for ML framework
import tensorflow as tf
"""
Sequential is the way to build model with the tf framework, we
can add layers, build, compile and fit the complete model
"""
from tensorflow.keras.models import Sequential
"""
Embedding layer - the bridge from events (represented as integers via IDs) and make
them vectors with fixed size - https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding
how to represent events to computers?
We can have a fixed amount of events, each one represented with vector full of zeros except one slot of the specific event
this is called one-hot encoding the shortcoming of this method is the lack of context between similar events on the system.
in addition, the data is very sparse but still in this method spend a lot of space.
with Embedding layer we represent an event with a vector full of real numbers (in most cases the vector has low dimensions)
during training this layer can learn when events occur in similar conditions and then the points on the vector space are become close to each other.
In this project the dimension of each vector is 16 and we have 175 evetns (matrix 175X16)
each row is a specific event (like hashtable) this vector is inputed to the LSTM layer

LSTM layer - Long Short Term Memory (Hochreiter 1997) represented the LSTM model that I covered in the article
https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM

Dense Layer - this is the normal NN connection when each neuron connected to each one of the next layer
we can choose the activation function (linear on defualt)
in this case for exaple the output layer is like this and we get vacab_size outputs (one for each event)
and output the unnormalize probability to each event to be the next

"""
from tensorflow.keras.layers import Embedding, LSTM, Dense


def build_ladohd_model(vocab_size=175, embedding_dim=16, hidden_size=64, seq_length=64):
    """
    Builds the LADOHD LSTM anomaly detection model.

    Args:
        vocab_size (int): Total number of unique system events(fixed number).
        embedding_dim (int): Dimension of the dense embedding vectors.
        hidden_size (int): Number of features in the LSTM hidden state.
        seq_length (int): Length of the input event sequences (BPTT window).

    Returns:
        tf.keras.Model: The compiled sequential model.
    """
    model = Sequential([
        # 1. Embedding Layer: Converts categorical event IDs into dense vectors
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=seq_length),

        # 2. LSTM Layers: 3 stacked layers to learn complex temporal patterns
        # return_sequences=True ensures the entire sequence is passed to the next layer
        LSTM(hidden_size, return_sequences=True),
        LSTM(hidden_size, return_sequences=True),
        LSTM(hidden_size, return_sequences=False),

        # 3. Fully Connected Layer: Extracts non-linear features from the LSTM outputs
        Dense(100, activation='relu'),

        # 4. Output Layer: Unnormalized log probabilities (logits) for each event in the vocabulary
        Dense(vocab_size)
    ])

    return model

if __name__ == "__main__":
    model = build_ladohd_model()

    # Compile the model with Adam optimizer and Sparse Categorical Crossentropy loss
    model.compile(
        optimizer='adam',
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

    model.summary()


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_10 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
import pandas as pd
import numpy as np
def load_and_filter_logs(file_path,target_actor="powershell.exe") -> pd.DataFrame:
    """
    Read raw EDR logs (CSV format) and filters them by specific actor.

    Args:
        file_path (str): path to the log file.
        target_actors (str, optional): The process name to monitor. Defaults to "powershell.exe".
    Returns:
        pd.DataFrame: A filtered DataFrame ordered by time.
    """
    #read the log file (in csv format)
    df = pd.read_csv(file_path)
    #sort the logs by the time they occured
    df = df.sort_values(by='Timestamp')
    #take just the rows that the actor is the one we filtering (and make a copy to not change the original)
    filtered_df = df[df['Actor'] == target_actor].copy()
    return filtered_df

def build_event_vocabulary(df,feature_coulmns):
    """
    Implements the transformation function F_T(.) to map raw features into a categorical vocabulary.

    Args:
        df (pd.DataFrame): The filtered logs.
        feature_columns (list): List of columns to define a unique event (e.g , ['EventType', 'Action', 'Target'] (coulmn names)).

    Returns:
        tuple: (List of sequential event IDs, Dictionary mapping feature tuples to IDs)
    """
    #for each combination of those coulmns make unique event
    unique_events = df[feature_coulmns].drop_duplicates()
    #map each event to unique ID
    vocab = {tuple(row) : idx for idx,row in enumerate(unique_events.values)}
    #make all the rows in the logs to sequence of ID's (continuous list) (the df is sorted by Timestamps)
    event_sequence  = [vocab[tuple(x)] for x in df[feature_coulmns].values]
    return event_sequence,vocab
def creating_training_sequence(event_sequence,seq_length=64):
    """
    Generates 3D tensors for LSTM training using a sliding window approach.

    Args:
        event_sequence (list): The full chronologial sequence of event IDs.
        seq_length (int): The BPTT unrolling window size (default: 64).

    Returns:
        tuple: (X_train numpy array, y_train numpy array)
    """
    X,y = [],[]
    #make sliding windows (each windows of events is with a length of seq_length )
    #the output is the next event each time (we want to give probability to the next event in real time)
    #if the event is with normal probability(top K probable events) (according to the connection the model learns-the training data is this windows)
    #this is benign action else this is anomalous action
    for i in range(len(event_sequence) - seq_length):
        X.append(event_sequence[i:i+seq_length])
        y.append(event_sequence[i+seq_length])
    return np.array(X),np.array(y)


In [20]:
#"Stop training when a monitored metric has stopped improving." - https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
from tensorflow.keras.callbacks import EarlyStopping


def train_ladohd_model(model, X_train, y_train, epochs=50, batch_size=64, validation_split=0.2):
    """
    Trains the LSTM model using the prepared sequences of benign events.

    Args:
        model: The compiled Keras sequential model.
        X_train (np.array): Input sequences (Sliding windows).
        y_train (np.array): Target next-events.
        epochs (int): Maximum number of training iterations.
        batch_size (int): Number of sequences to process before updating weights.
        validation_split (float): Fraction of data to use for validation.

    Returns:
        History object containing training metrics.
    """
    #define this process of earlyStopping to prevent overfitting
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )
    #see https://www.tensorflow.org/api_docs/python/tf/keras/Model#fit
    #X_train and y_train are the data to train on
    #epochs is the number of iteration over the whole dataset, the training can be stop before the number of epochs is the maximum (early_stopping)
    #batch_size is the number of samples that the model reads in one time (the mini-batch I explained)
    #validation_split takes the friction given from the dataset and uses it to validates the model (and not for training)
    #callbacks are list of function that performed during the trainings
    history =  model.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size = batch_size,
        validation_split= validation_split,
        callbacks= [early_stopping],
        verbose = 1
    )
    return history

In [21]:
import numpy as np
import tensorflow as tf
from scipy import stats

def detect_anomaly_dynamic(model, previous_sequence, actual_next_event):
    """
    Classifies an event as benign or anomalous using a dynamic threshold[cite: 1].

    Args:
        model: The trained LADOHD model.
        previous_sequence (np.array): The history window (e.g., length 64) of event IDs.
        actual_next_event (int): The ID of the event that actually occurred.

    Returns:
        bool: True if the event is anomalous, False if it is benign.
    """
    #make the tensor with one more dimension inorder to enable tf to process the mini-batch in diffrenct dimension.
    seq_tensor = np.expand_dims(previous_sequence, axis=0)

    #get raw unnomalized scores (logits) of all the possible vocab_size events , for every timestamp
    logits = model.predict(seq_tensor, verbose=0)

    #get the prediction just for final timestamp and apply the softmax function (convert each probability to number between 0 to 1 and the sum is 1)
    probabilities = tf.nn.softmax(logits[0, -1, :]).numpy()

    #round to 3 places decimals
    rounded_probs = np.round(probabilities, decimals=3)

    #we need to calculate the probability P(e_t | e_(t-1),e_(t-2) ... ) , for this we select K events that are most probable
    # (K is a parameter set staticly or dynamicly) if the next event is not from the K benign events this events is anomalous.
    #The static treshold is worse than the dynamic one in real life cases.
    #A method to make this treshold is to take the most repeated (mode) probability and make it a treshold.
    mode_result = stats.mode(rounded_probs, keepdims=False)
    dynamic_threshold = mode_result.mode
    #take the probability of the next event to occur
    actual_event_prob = probabilities[actual_next_event]
    #if lower than the treshold this is an anomanlous
    is_anomalous = actual_event_prob <= dynamic_threshold

    return is_anomalous



In [22]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf

# 1. טעינת קובץ המיילים שקיים בוודאות במאגר שהורדנו
email_path = os.path.join(path, "email.csv")
df_email = pd.read_csv(email_path)

# מיון כרונולוגי של האירועים על ציר הזמן
# משתמשים ב-format='mixed' כי תבניות התאריך עשויות להשתנות
df_email['date'] = pd.to_datetime(df_email['date'], format='mixed')
df_email = df_email.sort_values(by='date')

# 2. סינון משתמש ספציפי כדי ללמוד את שגרת העבודה הלגיטימית שלו
target_user = "LAP0338"
df_user = df_email[df_email['user'] == target_user].copy()

# מילוי ערכים ריקים במידה ויש מיילים ללא נמען מוגדר
df_user['to'] = df_user['to'].fillna('unknown')

# 3. בניית אוצר המילים (Vocabulary)
# אירוע במערכת מוגדר כעת כשילוב של המחשב (pc) והנמען (to)
features = ['pc', 'to']
event_sequence, vocab = build_event_vocabulary(df_user, features)
vocab_size = len(vocab)
print(f"Vocabulary size for {target_user}: {vocab_size} unique events.")

# 4. יצירת חלונות הזמן (BPTT Window)
X_train, y_train = creating_training_sequence(event_sequence, seq_length=10)
print(f"Training sequences shape: {X_train.shape}")

# 5. בניית המודל ואימון
model = build_ladohd_model(vocab_size=vocab_size, seq_length=10)
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

print("🚀 Starting Model Training...")
history = train_ladohd_model(model, X_train, y_train, epochs=20, batch_size=16)

Vocabulary size for LAP0338: 751 unique events.
Training sequences shape: (1410, 10)
🚀 Starting Model Training...
Epoch 1/20


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


71/71 ━━━━━━━━━━━━━━━━━━━━ 8s 30ms/step - accuracy: 0.0089 - loss: 6.5094 - val_accuracy: 0.0071 - val_loss: 6.4174
Epoch 2/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.0160 - loss: 6.1029 - val_accuracy: 0.0071 - val_loss: 6.7573
Epoch 3/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.0177 - loss: 6.0030 - val_accuracy: 0.0071 - val_loss: 7.1399
Epoch 4/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.0177 - loss: 5.9371 - val_accuracy: 0.0071 - val_loss: 7.2577
Epoch 5/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.0177 - loss: 5.8704 - val_accuracy: 0.0071 - val_loss: 7.4448
Epoch 6/20
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0195 - loss: 5.7777 - val_accuracy: 0.0071 - val_loss: 7.9047


In [23]:
# ---------------------------------------------------------
# סימולציית תקיפה וזיהוי אנומליות (Inference Phase)
# ---------------------------------------------------------

# בחירת רצף היסטורי מקרי מתוך נתוני הסביבה הלגיטימית (Benign)
sample_idx = 0
history_sequence = X_train[sample_idx]

# 1. תרחיש חוקי: האירוע שהמשתמש באמת ביצע מיד לאחר מכן
legitimate_event = y_train[sample_idx]

print(f"🔍 Testing Legitimate Event (ID: {legitimate_event})...")
alert_legitimate = detect_anomaly_dynamic(model, history_sequence, legitimate_event)

if alert_legitimate:
    print("❌ False Alarm: Legitimate event was flagged as an anomaly.")
else:
    print("✅ Success: Legitimate event passed the dynamic threshold naturally.")

print("-" * 50)

# 2. תרחיש מתקפה (Insider Threat): התחברות חריגה למשאב לא מוכר
# ניקח את המזהה האחרון במילון (לרוב מייצג שרת או פעולה נדירה מאוד למשתמש הזה)
malicious_event = vocab_size - 1

print(f"🚨 Testing Malicious Event (ID: {malicious_event})...")
alert_malicious = detect_anomaly_dynamic(model, history_sequence, malicious_event)

if alert_malicious:
    print("🎯 Threat Neutralized: Malicious event successfully detected and blocked!")
else:
    print("⚠️ Failure: Malicious event slipped through the model.")

🔍 Testing Legitimate Event (ID: 10)...


IndexError: too many indices for array: array is 2-dimensional, but 3 were indexed